<a href="https://colab.research.google.com/github/teklemaryamas-et/HWRM_Thesis_2026/blob/main/NEX_GDDP_CMIP6_Future_FULL_REDONE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# FULL MIROC6 FUTURE DOWNLOAD FROM GEE / GOOGLE EARTH ENGINE
# Model: MIROC6
# Scenarios: ssp245 and ssp585
# Period: 2030–2080
# Variable: pr
# Output unit: monthly rainfall total, mm/month
# Study area: Upper Blue Nile Basin / Abay
# ============================================================

# ============================================================
# 1. INSTALL PACKAGES
# ============================================================

!pip install -q earthengine-api geemap rasterio pandas

import ee
import geemap
import os
import calendar
import pandas as pd
import rasterio
import numpy as np
from google.colab import drive

# ============================================================
# 2. AUTHENTICATE AND INITIALIZE EARTH ENGINE
# ============================================================

ee.Authenticate()

# Your Google Cloud project
ee.Initialize(project="velvety-being-497812-e5")

# ============================================================
# 3. MOUNT GOOGLE DRIVE
# ============================================================

drive.mount("/content/drive")

# ============================================================
# 4. OUTPUT FOLDER
# ============================================================

# New clean folder. Do not overwrite your old folder.
base_out_dir = "/content/drive/MyDrive/NEX_GDDP_CMIP6_Future_FULL_REDONE/MIROC6"

os.makedirs(base_out_dir, exist_ok=True)

print("Output folder:")
print(base_out_dir)

# ============================================================
# 5. STUDY AREA / BASIN BOUNDARY
# ============================================================

# Your Abay / Upper Blue Nile Basin asset
basin = ee.FeatureCollection("projects/velvety-being-497812-e5/assets/abay")

# Use basin geometry
basin_geom = basin.geometry()

# Use basin bounds for export region to avoid partial export problems
export_region = basin_geom.bounds()

# ============================================================
# 6. SETTINGS
# ============================================================

model = "MIROC6"
variable = "pr"

scenarios = ["ssp245", "ssp585"]

start_year = 2030
end_year = 2080

# Export scale in meters.
# NEX-GDDP-CMIP6 is about 0.25 degree, so 25000 m is appropriate.
export_scale = 25000

# Expected total files
expected_files_per_scenario = (end_year - start_year + 1) * 12
expected_total_files = expected_files_per_scenario * len(scenarios)

print("Expected files per scenario:", expected_files_per_scenario)
print("Expected total files:", expected_total_files)

# ============================================================
# 7. FUNCTION TO CREATE MONTHLY RAINFALL IMAGE
# ============================================================

def get_monthly_pr_mm(model, scenario, year, month):
    """
    NEX-GDDP-CMIP6 pr is precipitation flux.
    Monthly rainfall total = sum(daily pr) * 86400
    Unit becomes approximately mm/month.
    """

    start_date = f"{year}-{month:02d}-01"

    if month == 12:
        end_date = f"{year + 1}-01-01"
    else:
        end_date = f"{year}-{month + 1:02d}-01"

    ic = (
        ee.ImageCollection("NASA/GDDP-CMIP6")
        .filter(ee.Filter.eq("model", model))
        .filter(ee.Filter.eq("scenario", scenario))
        .filterDate(start_date, end_date)
        .select(variable)
    )

    image_count = ic.size().getInfo()

    if image_count == 0:
        raise ValueError(f"No GEE images found for {model}, {scenario}, {year}-{month:02d}")

    monthly_img = (
        ic.sum()
        .multiply(86400)
        .rename("pr")
        .clip(export_region)
        .set({
            "model": model,
            "scenario": scenario,
            "year": year,
            "month": month,
            "unit": "mm/month",
            "daily_image_count": image_count
        })
    )

    return monthly_img, image_count

# ============================================================
# 8. FUNCTION TO CHECK DOWNLOADED TIF VALIDITY
# ============================================================

def check_tif_validity(tif_path):
    """
    Returns valid cell count and min/max/mean.
    """
    try:
        with rasterio.open(tif_path) as src:
            arr = src.read(1)
            nodata = src.nodata

            if nodata is not None:
                arr = np.where(arr == nodata, np.nan, arr)

            valid = np.isfinite(arr)
            valid_count = int(np.sum(valid))

            if valid_count == 0:
                return {
                    "valid_count": 0,
                    "min": np.nan,
                    "max": np.nan,
                    "mean": np.nan,
                    "status": "empty_or_all_nan"
                }

            return {
                "valid_count": valid_count,
                "min": float(np.nanmin(arr)),
                "max": float(np.nanmax(arr)),
                "mean": float(np.nanmean(arr)),
                "status": "valid"
            }

    except Exception as e:
        return {
            "valid_count": 0,
            "min": np.nan,
            "max": np.nan,
            "mean": np.nan,
            "status": f"read_error: {e}"
        }

# ============================================================
# 9. DOWNLOAD ALL MONTHS
# ============================================================

download_log = []

for scenario in scenarios:

    for year in range(start_year, end_year + 1):

        # Folder structure similar to your previous workflow
        if 2030 <= year <= 2050:
            period_name = "Near_future_2030_2050"
        else:
            period_name = "Far_future_2051_2080"

        scenario_dir = os.path.join(base_out_dir, scenario, period_name)
        os.makedirs(scenario_dir, exist_ok=True)

        for month in range(1, 13):

            filename = f"{model}_{scenario}_{year}_{month:02d}.tif"
            out_file = os.path.join(scenario_dir, filename)

            print("\n================================================")
            print("Downloading:", filename)
            print("================================================")

            # If file already exists, check it first
            if os.path.exists(out_file):
                check = check_tif_validity(out_file)
                if check["status"] == "valid" and check["valid_count"] > 0:
                    print("Already exists and valid. Skipping:", out_file)
                    download_log.append({
                        "model": model,
                        "scenario": scenario,
                        "period": period_name,
                        "year": year,
                        "month": month,
                        "filename": filename,
                        "daily_image_count": None,
                        "output_file": out_file,
                        "valid_count": check["valid_count"],
                        "min": check["min"],
                        "max": check["max"],
                        "mean": check["mean"],
                        "status": "already_exists_valid"
                    })
                    continue
                else:
                    print("Existing file is invalid. Deleting and redownloading.")
                    os.remove(out_file)

            status = "failed"
            image_count = None
            check = {
                "valid_count": 0,
                "min": np.nan,
                "max": np.nan,
                "mean": np.nan,
                "status": "not_checked"
            }

            # Retry up to 3 times
            for attempt in range(1, 4):
                try:
                    print(f"Attempt {attempt} for {filename}")

                    image, image_count = get_monthly_pr_mm(model, scenario, year, month)

                    geemap.ee_export_image(
                        image,
                        filename=out_file,
                        scale=export_scale,
                        region=export_region,
                        file_per_band=False
                    )

                    check = check_tif_validity(out_file)

                    if check["status"] == "valid" and check["valid_count"] > 0:
                        status = "downloaded_valid"
                        print("Downloaded valid file:", out_file)
                        print("Valid cells:", check["valid_count"])
                        print("Min:", check["min"], "Max:", check["max"], "Mean:", check["mean"])
                        break
                    else:
                        status = f"downloaded_but_invalid_attempt_{attempt}"
                        print("Downloaded but invalid. Retrying...")
                        if os.path.exists(out_file):
                            os.remove(out_file)

                except Exception as e:
                    status = f"failed_attempt_{attempt}: {e}"
                    print("Error:", e)
                    if os.path.exists(out_file):
                        try:
                            os.remove(out_file)
                        except:
                            pass

            download_log.append({
                "model": model,
                "scenario": scenario,
                "period": period_name,
                "year": year,
                "month": month,
                "filename": filename,
                "daily_image_count": image_count,
                "output_file": out_file,
                "valid_count": check["valid_count"],
                "min": check["min"],
                "max": check["max"],
                "mean": check["mean"],
                "status": status
            })

# ============================================================
# 10. SAVE DOWNLOAD LOG
# ============================================================

log_df = pd.DataFrame(download_log)

log_csv = os.path.join(base_out_dir, "MIROC6_future_full_download_log.csv")
log_df.to_csv(log_csv, index=False)

print("\n================================================")
print("DOWNLOAD FINISHED")
print("================================================")
print("Download log saved at:")
print(log_csv)

print("\nStatus summary:")
print(log_df["status"].value_counts())

print("\nFiles downloaded/checked:", len(log_df))
print("Expected total files:", expected_total_files)

# ============================================================
# 11. FINAL VALIDATION SUMMARY
# ============================================================

bad_df = log_df[
    (log_df["valid_count"] <= 0) |
    (~log_df["status"].astype(str).str.contains("valid"))
]

bad_csv = os.path.join(base_out_dir, "BAD_OR_INVALID_FILES_AFTER_DOWNLOAD.csv")
bad_df.to_csv(bad_csv, index=False)

print("\nNumber of bad/invalid files:", len(bad_df))

if len(bad_df) > 0:
    print("Bad/invalid files saved at:")
    print(bad_csv)
    print(bad_df[["scenario", "year", "month", "filename", "status", "valid_count"]])
else:
    print("All downloaded files are valid.")

# Count files physically present
all_tifs = []
for root, dirs, files in os.walk(base_out_dir):
    for f in files:
        if f.lower().endswith((".tif", ".tiff")):
            all_tifs.append(os.path.join(root, f))

print("\nPhysical tif files found:", len(all_tifs))
print("Expected tif files:", expected_total_files)

if len(bad_df) == 0 and len(all_tifs) == expected_total_files:
    print("\nRESULT: Full MIROC6 future dataset downloaded successfully and is valid.")
else:
    print("\nRESULT: Download completed, but some files need checking.")

Streaming output truncated to the last 5000 lines.
Downloaded valid file: /content/drive/MyDrive/NEX_GDDP_CMIP6_Future_FULL_REDONE/MIROC6/ssp585/Near_future_2030_2050/MIROC6_ssp585_2046_09.tif
Valid cells: 575
Min: 147.97507500479696 Max: 558.8140071234193 Mean: 369.8997513656387

Downloading: MIROC6_ssp585_2046_10.tif
Attempt 1 for MIROC6_ssp585_2046_10.tif
Generating URL ...
Please wait ...
Data downloaded to /content/drive/MyDrive/NEX_GDDP_CMIP6_Future_FULL_REDONE/MIROC6/ssp585/Near_future_2030_2050/MIROC6_ssp585_2046_10.tif
Downloaded valid file: /content/drive/MyDrive/NEX_GDDP_CMIP6_Future_FULL_REDONE/MIROC6/ssp585/Near_future_2030_2050/MIROC6_ssp585_2046_10.tif
Valid cells: 575
Min: 0.4024893969472032 Max: 246.00225195972598 Mean: 66.69076375952547

Downloading: MIROC6_ssp585_2046_11.tif
Attempt 1 for MIROC6_ssp585_2046_11.tif
Generating URL ...
Please wait ...
Data downloaded to /content/drive/MyDrive/NEX_GDDP_CMIP6_Future_FULL_REDONE/MIROC6/ssp585/Near_future_2030_2050/MIROC6_s